In [ ]:
from jupyter_plotly_dash import JupyterDash

import dash
import dash_leaflet as dl
from dash import dcc, html, dash_table
import plotly.express as px
from dash.dependencies import Input, Output, State

import os
import numpy as np
import pandas as pd
from pymongo import MongoClient
from bson.json_util import dumps
import base64

import animalShelter_enhanced
from animalShelter_enhanced import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "aacuser"
animals = AnimalShelter(username, password)


# class read method must support return of cursor object 
df = pd.DataFrame.from_records(animals.read({}))

############################################
# Algorithms and Data Structures Enhancement
############################################

# A dictionary stores each rescue category and its MongoDB query.
# This replaces repeated conditional query code and makes it easier
# to add or modify rescue categories.
RESCUE_FILTERS = {
    "water": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]
        },
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {
            "$gte": 26.0,
            "$lte": 156.0
        }
    },

    "mount": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]
        },
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {
            "$gte": 26.0,
            "$lte": 156.0
        }
    },

    "disaster": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]
        },
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {
            "$gte": 20.0,
            "$lte": 300.0
        }
    },

    "reset": {}
}


def build_search_query(
    rescue_type,
    breed_text=None,
    sex_filter=None,
    minimum_age=None,
    maximum_age=None
):
    """
    Build a MongoDB query from the user's selected search criteria.

    The query is stored in a dictionary because MongoDB accepts
    dictionaries for filtering records.
    """
    query = RESCUE_FILTERS.get(rescue_type, {}).copy()

    # Add a partial, case-insensitive breed search.
    if breed_text and breed_text.strip():
        query["breed"] = {
            "$regex": breed_text.strip(),
            "$options": "i"
        }

    # Add a sex filter when the user selects one.
    if sex_filter and sex_filter != "all":
        query["sex_upon_outcome"] = sex_filter

    # Build an age-range dictionary only when an age is provided.
    age_range = {}

    if minimum_age is not None:
        age_range["$gte"] = float(minimum_age)

    if maximum_age is not None:
        age_range["$lte"] = float(maximum_age)

    if age_range:
        query["age_upon_outcome_in_weeks"] = age_range

    return query


def sort_records(records, sort_field, sort_direction):
    """
    Sort records while keeping missing values at the end.
    """
    if not records or not sort_field:
        return records

    reverse_sort = str(sort_direction).lower() == "descending"

    records_with_values = []
    records_without_values = []

    for record in records:
        value = record.get(sort_field)

        if value is None or value == "":
            records_without_values.append(record)
        else:
            records_with_values.append(record)

    def safe_sort_key(record):
        value = record.get(sort_field)

        if isinstance(value, (int, float)) and not isinstance(value, bool):
            return (0, float(value))

        return (1, str(value).lower())

    sorted_records = sorted(
        records_with_values,
        key=safe_sort_key,
        reverse=reverse_sort
    )

    return sorted_records + records_without_values

#########################
# Dashboard Layout / View
#########################
# create dash application

external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)


image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())
                               
app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('SNHU CS-499'))),
    html.Center(html.B(html.H1('Grazioso Salvare Animal Rescue Dashboard'))),
    html.Hr(),
    html.Img(id='customer-image',src='data:image/png;base64,{}'.format(encoded_image.decode()),alt='customer image'),
    html.Div([
    html.H3("Animal Search and Sorting Controls"),

    html.Label("Rescue Category"),
    dcc.Dropdown(
        id="filter-type",
        options=[
            {"label": "All Animals", "value": "reset"},
            {"label": "Water Rescue", "value": "water"},
            {
                "label": "Mountain/Wilderness Rescue",
                "value": "mount"
            },
            {
                "label": "Disaster Rescue and Individual Tracking",
                "value": "disaster"
            }
        ],
        value="reset",
        clearable=False
    ),

    html.Br(),

    html.Label("Breed Search"),
    dcc.Input(
        id="breed-search",
        type="text",
        placeholder="Enter all or part of a breed",
        value=""
    ),

    html.Br(),
    html.Br(),

    html.Label("Sex Upon Outcome"),
    dcc.Dropdown(
        id="sex-filter",
        options=[
            {"label": "All", "value": "all"},
            {"label": "Intact Female", "value": "Intact Female"},
            {"label": "Spayed Female", "value": "Spayed Female"},
            {"label": "Intact Male", "value": "Intact Male"},
            {"label": "Neutered Male", "value": "Neutered Male"},
            {"label": "Unknown", "value": "Unknown"}
        ],
        value="all",
        clearable=False
    ),

    html.Br(),

    html.Div([
        html.Div([
            html.Label("Minimum Age in Weeks"),
            dcc.Input(
                id="minimum-age",
                type="number",
                min=0,
                value=None,
                placeholder="Minimum age"
            )
        ], style={
            "display": "inline-block",
            "width": "48%"
        }),

        html.Div([
            html.Label("Maximum Age in Weeks"),
            dcc.Input(
                id="maximum-age",
                type="number",
                min=0,
                value=None,
                placeholder="Maximum age"
            )
        ], style={
            "display": "inline-block",
            "width": "48%"
        })
    ]),

    html.Br(),
    html.Br(),

    html.Div([
        html.Div([
            html.Label("Sort Records By"),
            dcc.Dropdown(
                id="sort-field",
                options=[
                    {
                        "label": "Animal Name",
                        "value": "name"
                    },
                    {
                        "label": "Breed",
                        "value": "breed"
                    },
                    {
                        "label": "Age in Weeks",
                        "value": "age_upon_outcome_in_weeks"
                    },
                    {
                        "label": "Sex Upon Outcome",
                        "value": "sex_upon_outcome"
                    },
                    {
                        "label": "Animal Type",
                        "value": "animal_type"
                    }
                ],
                value="name",
                clearable=False
            )
        ], style={
            "display": "inline-block",
            "width": "48%"
        }),

        html.Div([
            html.Label("Sort Direction"),
            dcc.RadioItems(
                id="sort-direction",
                options=[
                    {
                        "label": "Ascending",
                        "value": "ascending"
                    },
                    {
                        "label": "Descending",
                        "value": "descending"
                    }
                ],
                value="ascending",
                labelStyle={
                    "display": "inline-block",
                    "marginRight": "15px"
                }
            )
        ], style={
            "display": "inline-block",
            "width": "48%",
            "verticalAlign": "top"
        })
    ]),

    html.Br(),

    html.Div(
        id="results-message",
        style={
            "fontWeight": "bold",
            "marginTop": "10px"
        }
    )
]),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        # Allows 10 rows per page, and starts on page 0. Columns and Rows may be selected but not deleted/edited
        editable=False,
        sort_action="native",
        sort_mode="multi",
        column_selectable=True,
        row_selectable=True,
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current= 0,
        page_size= 10,
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id="map-id",
            className="col s12 m6",
            style={
                "width": "48%",
                "minHeight": "500px",
                "display": "inline-block",
                "verticalAlign": "top"
            }
        )
    ])
])

def retrieve_filtered_records(query):
    """
    Retrieve matching animal records from MongoDB.
    """
    return list(animals.read(query))

#############################################
# Interaction Between Components / Controller
#############################################

# Enhanced filtering and sorting callback
@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "columns"),
        Output("datatable-id", "selected_rows"),
        Output("results-message", "children")
    ],
    [
        Input("filter-type", "value"),
        Input("breed-search", "value"),
        Input("sex-filter", "value"),
        Input("minimum-age", "value"),
        Input("maximum-age", "value"),
        Input("sort-field", "value"),
        Input("sort-direction", "value")
    ]
)
def update_dashboard(
    rescue_type,
    breed_text,
    sex_filter,
    minimum_age,
    maximum_age,
    sort_field,
    sort_direction,
    **kwargs
):
    """
    Update the dashboard using multiple search criteria and sorting.
    """
    try:
        # Prevent an invalid age range.
        if (
            minimum_age is not None
            and maximum_age is not None
            and minimum_age > maximum_age
        ):
            return (
                [],
                [],
                [],
                "Minimum age cannot be greater than maximum age."
            )

        query = build_search_query(
            rescue_type,
            breed_text,
            sex_filter,
            minimum_age,
            maximum_age
        )

        records = retrieve_filtered_records(query)

        records = sort_records(
            records,
            sort_field,
            sort_direction
        )

        if not records:
            return (
                [],
                [],
                [],
                "No animal records matched the selected criteria."
            )

        filtered_df = pd.DataFrame.from_records(records)

        columns = [
            {
                "name": column,
                "id": column,
                "deletable": False,
                "selectable": True
            }
            for column in filtered_df.columns
        ]

        data = filtered_df.to_dict("records")

        return (
            data,
            columns,
            [0],
            f"{len(data)} matching animal record(s) found."
        )

    except Exception as error:
        return (
            [],
            [],
            [],
            f"Unable to retrieve records: {error}"
        )

# Change background color of selected columns
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# Chart
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_viewport_data")]
)
def update_graphs(view_data):
    """
    Create a breed distribution chart from the visible table records.
    """
    if not view_data:
        return [
            html.P(
                "No chart is available because no records matched "
                "the selected criteria."
            )
        ]

    chart_df = pd.DataFrame.from_records(view_data)

    if "breed" not in chart_df.columns:
        return [html.P("Breed data is unavailable.")]

    breed_counts = (
        chart_df["breed"]
        .fillna("Unknown")
        .value_counts()
        .reset_index()
    )

    breed_counts.columns = ["breed", "count"]

    figure = px.bar(
        breed_counts,
        x="breed",
        y="count",
        title="Visible Animal Records by Breed",
        labels={
            "breed": "Breed",
            "count": "Number of Animals"
        }
    )

    return [dcc.Graph(figure=figure)]

# Map
# Map callback
@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "data"),
        Input("datatable-id", "selected_rows")
    ]
)
def update_map(table_data, selected_rows, **kwargs):
    """
    Display the selected animal's location on the map.
    """
    if not table_data:
        return html.P(
            "No map is available because no records matched "
            "the selected criteria."
        )

    map_df = pd.DataFrame.from_records(table_data)

    required_columns = {
        "location_lat",
        "location_long",
        "breed",
        "name"
    }

    if not required_columns.issubset(map_df.columns):
        return html.P("Location information is unavailable.")

    selected_index = 0

    if selected_rows:
        selected_index = selected_rows[-1]

    if selected_index >= len(map_df):
        selected_index = 0

    selected_animal = map_df.iloc[selected_index]

    latitude = selected_animal["location_lat"]
    longitude = selected_animal["location_long"]

    if pd.isna(latitude) or pd.isna(longitude):
        return html.P("The selected animal has no location data.")

    latitude = float(latitude)
    longitude = float(longitude)

    animal_name = selected_animal.get("name", "Unknown")
    animal_breed = selected_animal.get("breed", "Unknown")

    if pd.isna(animal_name) or animal_name == "":
        animal_name = "Unknown"

    if pd.isna(animal_breed) or animal_breed == "":
        animal_breed = "Unknown"

    return dl.Map(
        center=[latitude, longitude],
        zoom=10,
        style={
            "width": "100%",
            "height": "500px"
        },
        children=[
            dl.TileLayer(),
            dl.Marker(
                position=[latitude, longitude],
                children=[
                    dl.Tooltip(str(animal_breed)),
                    dl.Popup([
                        html.H4("Animal Name"),
                        html.P(str(animal_name)),
                        html.P(f"Breed: {animal_breed}")
                    ])
                ]
            )
        ]
    )


if __name__ == '__main__':
    app.run(debug=False)